# Lab 09: Model Comparison and Benchmarks (Gemini 3.1)

Choosing the right model is a trade-off between **Intelligence**, **Latency**, and **Cost**. In this lab, we will benchmark **Gemini 3.1 Flash-Lite** against **Gemini 3.1 Pro** across these three dimensions.

## Objectives
1. Measure and compare **Latency** (TTFT and Total Time).
2. Evaluate **Reasoning Capabilities** using a logic puzzle.
3. Implement a **Cost Calculator** based on real token usage.

In [ ]:
import os
import time

from dotenv import load_dotenv
from google import genai
from IPython.display import Markdown, display

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

MODELS = {
    "flash": "gemini-3.1-flash-lite-preview",
    "pro": "gemini-3.1-pro-preview"
}

print("Client initialized.")

## 1. Latency Benchmark

We will measure the **Time to First Token (TTFT)** using streaming, as it is the most critical metric for user experience.

In [ ]:
def benchmark_latency(model_name, prompt):
    print(f"--- Benchmarking {model_name} ---")
    start_time = time.perf_counter()
    ttft = None

    # Use '_' for the loop variable to avoid Ruff warnings about unused variables
    for _ in client.models.generate_content_stream(
        model=model_name,
        contents=prompt
    ):
        if ttft is None:
            ttft = time.perf_counter() - start_time

    total_time = time.perf_counter() - start_time
    print(f"TTFT: {ttft:.4f}s")
    print(f"Total Time: {total_time:.4f}s\n")
    return ttft, total_time

prompt = "Write a 200-word essay on the importance of space exploration."
benchmark_latency(MODELS['flash'], prompt)
benchmark_latency(MODELS['pro'], prompt)

## 2. Reasoning Benchmark

Can Flash-Lite solve complex logic? Let's compare its answer with the Pro model.

In [ ]:
logic_puzzle = """
Sally has 3 brothers. Each of her brothers has 2 sisters.
How many sisters does Sally have? Explain your reasoning step by step.
"""

for key, model_id in MODELS.items():
    print(f"### Results for {key.upper()}:")
    response = client.models.generate_content(model=model_id, contents=logic_puzzle)
    display(Markdown(response.text))
    print("-" * 30)

## 3. Cost Calculator

Let's implement a function to estimate the cost of a request based on approximate pricing (e.g., March 2026 rates).

In [ ]:
# Example pricing per 1M tokens (Simulated for March 2026)
PRICING = {
    MODELS['flash']: {"input": 0.10, "output": 0.30}, # Very cheap
    MODELS['pro']: {"input": 1.25, "output": 3.75}    # More expensive
}

def calculate_cost(model_name, response):
    usage = response.usage_metadata
    input_tokens = usage.prompt_token_count
    output_tokens = usage.candidates_token_count

    rates = PRICING.get(model_name)
    cost = (
        input_tokens / 1_000_000 * rates['input']
    ) + (
        output_tokens / 1_000_000 * rates['output']
    )

    print(f"Model: {model_name}")
    print(f"Tokens: In={input_tokens}, Out={output_tokens}")
    print(f"Estimated Cost: ${cost:.6f}\n")
    return cost

prompt = "Analyze the impact of interest rates on the global economy in 500 words."
for model_id in MODELS.values():
    resp = client.models.generate_content(model=model_id, contents=prompt)
    calculate_cost(model_id, resp)

## Summary

Through these benchmarks, you've learned:
1. **Flash-Lite** is significantly faster (lower TTFT) and cheaper, making it ideal for UI-driven or high-volume tasks.
2. **Pro** offers superior reasoning for complex logic puzzles.
3. **Token Monitoring** is essential for managing production budgets.